In [ ]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas numpy matplotlib tqdm

In [ ]:
import transformers
import datasets
import evaluate
import torch
import sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("evaluate:", evaluate.__version__)
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Deep Learning Question Answering for Document Intelligence

This notebook builds an extractive QA pipeline using transformer models, evaluates a pretrained baseline, fine-tunes DistilBERT on SQuAD, and compares Exact Match (EM) and F1.

In [ ]:
from pathlib import Path
import json
import re
import string
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)

PROJECT_NAME = "qa_transformer_project"

print("Project:", PROJECT_NAME)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
print("Loading SQuAD...")
dataset = load_dataset("squad")

train_ds = dataset["train"]
val_ds = dataset["validation"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
print("Columns:", train_ds.column_names)

sample = train_ds[0]
print("\nSample example:")
for k, v in sample.items():
    print(f"\n--- {k} ---")
    print(v)

In [ ]:
summary = {
    "train_size": len(train_ds),
    "validation_size": len(val_ds),
    "columns": train_ds.column_names,
}

summary

## 2. Pretrained Baseline QA Model

In [ ]:
def answer_question(tokenizer, model, question, context, max_length=512, max_answer_length=30):
    """
    Run extractive QA for one question-context pair using a constrained best-span search.
    """
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    input_ids = inputs["input_ids"][0]

    best_score = float("-inf")
    best_start = 0
    best_end = 0

    for start_idx in range(len(start_logits)):
        max_end_idx = min(start_idx + max_answer_length, len(end_logits))
        for end_idx in range(start_idx, max_end_idx):
            score = start_logits[start_idx].item() + end_logits[end_idx].item()
            if score > best_score:
                best_score = score
                best_start = start_idx
                best_end = end_idx

    answer_tokens = input_ids[best_start : best_end + 1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

    start_prob = torch.softmax(start_logits, dim=0)[best_start].item()
    end_prob = torch.softmax(end_logits, dim=0)[best_end].item()
    confidence = (start_prob + end_prob) / 2.0

    return {
        "answer": answer,
        "score": confidence,
        "start_idx": best_start,
        "end_idx": best_end,
    }

In [ ]:
BASELINE_MODEL_NAME = "bert-large-uncased-whole-word-masking-finetuned-squad"

print("Loading baseline model:", BASELINE_MODEL_NAME)

baseline_tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME)
baseline_model = AutoModelForQuestionAnswering.from_pretrained(BASELINE_MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
baseline_model.to(device)
baseline_model.eval()

print("Baseline model loaded.")
print("Device:", device)

In [ ]:
baseline_results = []

for i in range(5):
    example = val_ds[i]

    question = example["question"]
    context = example["context"]
    gold_answer = example["answers"]["text"][0]

    pred = answer_question(
        tokenizer=baseline_tokenizer,
        model=baseline_model,
        question=question,
        context=context,
    )

    baseline_results.append({
        "question": question,
        "gold_answer": gold_answer,
        "predicted_answer": pred["answer"],
        "confidence": pred["score"],
    })

    print("\n----------------------------------------")
    print("Question:", question)
    print("Gold:", gold_answer)
    print("Predicted:", pred["answer"])
    print("Confidence:", round(pred["score"], 4))

## 3. Baseline Evaluation: Exact Match (EM) and F1

In [ ]:
def normalize_answer(s):
    """
    Lower text and remove punctuation, articles, and extra whitespace.
    """
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))


def compute_f1(a_gold, a_pred):
    gold_toks = normalize_answer(a_gold).split()
    pred_toks = normalize_answer(a_pred).split()

    common = Counter(gold_toks) & Counter(pred_toks)
    num_same = sum(common.values())

    if len(gold_toks) == 0 or len(pred_toks) == 0:
        return float(gold_toks == pred_toks)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_toks)
    recall = num_same / len(gold_toks)
    return 2 * precision * recall / (precision + recall)

In [ ]:
n_eval = 100

baseline_eval_results = []
baseline_em_scores = []
baseline_f1_scores = []

for i in range(n_eval):
    example = val_ds[i]

    question = example["question"]
    context = example["context"]
    gold_answers = example["answers"]["text"]

    pred = answer_question(
        tokenizer=baseline_tokenizer,
        model=baseline_model,
        question=question,
        context=context,
    )

    pred_answer = pred["answer"]

    em = max(compute_exact(gold, pred_answer) for gold in gold_answers)
    f1 = max(compute_f1(gold, pred_answer) for gold in gold_answers)

    baseline_em_scores.append(em)
    baseline_f1_scores.append(f1)

    baseline_eval_results.append({
        "id": example["id"],
        "question": question,
        "predicted_answer": pred_answer,
        "gold_answers": gold_answers,
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

baseline_em = 100 * np.mean(baseline_em_scores)
baseline_f1 = 100 * np.mean(baseline_f1_scores)

print("Baseline QA Evaluation")
print("======================")
print(f"Examples evaluated : {n_eval}")
print(f"Exact Match (EM)   : {baseline_em:.2f}")
print(f"F1 Score           : {baseline_f1:.2f}")

In [ ]:
baseline_eval_df = pd.DataFrame(baseline_eval_results)
baseline_eval_df.head(10)

## 4. Fine-Tune DistilBERT for Extractive QA

In [ ]:
FINETUNE_MODEL_NAME = "bert-base-uncased"

print("Loading fine-tuning model:", FINETUNE_MODEL_NAME)

finetune_tokenizer = AutoTokenizer.from_pretrained(FINETUNE_MODEL_NAME)
finetune_model = AutoModelForQuestionAnswering.from_pretrained(FINETUNE_MODEL_NAME)

finetune_model.to(device)

print("Fine-tuning model loaded.")
print("Device:", device)

In [ ]:
MAX_LENGTH = 384
DOC_STRIDE = 128

def prepare_train_features(examples, tokenizer):
    questions = [q.strip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        if len(answers["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        start_char = answers["answer_start"][0]
        end_char = start_char + len(answers["text"][0])

        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        token_end_index = len(input_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        if not (
            offsets[token_start_index][0] <= start_char
            and offsets[token_end_index][1] >= end_char
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            while (
                token_start_index < len(offsets)
                and offsets[token_start_index][0] <= start_char
            ):
                token_start_index += 1
            start_positions.append(token_start_index - 1)

            while offsets[token_end_index][1] >= end_char:
                token_end_index -= 1
            end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples

In [ ]:
train_subset = train_ds.select(range(20000))
val_subset = val_ds.select(range(1000))

print("Train subset size:", len(train_subset))
print("Validation subset size:", len(val_subset))

In [ ]:
tokenized_train = train_subset.map(
    lambda x: prepare_train_features(x, finetune_tokenizer),
    batched=True,
    remove_columns=train_subset.column_names,
)

tokenized_val = val_subset.map(
    lambda x: prepare_train_features(x, finetune_tokenizer),
    batched=True,
    remove_columns=val_subset.column_names,
)

print("Tokenized train size:", len(tokenized_train))
print("Tokenized validation size:", len(tokenized_val))

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./outputs/models/bert_base_qa_final",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=finetune_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=default_data_collator,
)

In [ ]:
train_result = trainer.train()
train_result

## 5. Evaluate the Fine-Tuned DistilBERT Model

In [ ]:
n_eval = 100

finetuned_eval_results = []
finetuned_em_scores = []
finetuned_f1_scores = []

finetune_model.eval()

for i in range(n_eval):
    example = val_ds[i]

    question = example["question"]
    context = example["context"]
    gold_answers = example["answers"]["text"]

    pred = answer_question(
        tokenizer=finetune_tokenizer,
        model=finetune_model,
        question=question,
        context=context,
    )

    pred_answer = pred["answer"]

    em = max(compute_exact(gold, pred_answer) for gold in gold_answers)
    f1 = max(compute_f1(gold, pred_answer) for gold in gold_answers)

    finetuned_em_scores.append(em)
    finetuned_f1_scores.append(f1)

    finetuned_eval_results.append({
        "id": example["id"],
        "question": question,
        "predicted_answer": pred_answer,
        "gold_answers": gold_answers,
        "confidence": pred["score"],
        "exact_match": em,
        "f1": f1,
    })

finetuned_em = 100 * np.mean(finetuned_em_scores)
finetuned_f1 = 100 * np.mean(finetuned_f1_scores)

print("Fine-Tuned DistilBERT Evaluation")
print("================================")
print(f"Examples evaluated : {n_eval}")
print(f"Exact Match (EM)   : {finetuned_em:.2f}")
print(f"F1 Score           : {finetuned_f1:.2f}")

In [ ]:
finetuned_eval_df = pd.DataFrame(finetuned_eval_results)
finetuned_eval_df.head(10)

In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "Pretrained BERT QA",
        "exact_match": baseline_em,
        "f1": baseline_f1
    },
    {
        "model": "Fine-Tuned DistilBERT QA",
        "exact_match": finetuned_em,
        "f1": finetuned_f1
    }
])

comparison_df

In [ ]:
comparison_long = comparison_df.melt(
    id_vars="model",
    value_vars=["exact_match", "f1"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(8, 5))
for metric in ["exact_match", "f1"]:
    subset = comparison_long[comparison_long["metric"] == metric]
    plt.plot(subset["model"], subset["score"], marker="o", label=metric)

plt.ylabel("Score")
plt.title("Baseline vs Fine-Tuned QA Performance")
plt.legend()
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)
plt.show()

## 6. Error Analysis: Why Did Fine-Tuning Degrade Performance?

In [ ]:
error_cases = finetuned_eval_df[
    (finetuned_eval_df["exact_match"] == 0) &
    (finetuned_eval_df["f1"] < 0.5)
]

error_cases[["question", "predicted_answer", "gold_answers", "confidence"]].head(10)

In [ ]:
import os
import pandas as pd

# Create folders
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/tables", exist_ok=True)

# Save model comparison summary
summary_df = pd.DataFrame({
    "model": ["Pretrained BERT QA", "Fine-Tuned DistilBERT QA"],
    "exact_match": [84.0, 71.0],
    "f1": [90.64, 80.32]
})
summary_df.to_csv("outputs/tables/model_comparison.csv", index=False)

# ✅ Save FINAL (fine-tuned) predictions
pd.DataFrame(finetuned_eval_results).to_csv(
    "outputs/tables/sample_predictions.csv",
    index=False
)

# Save plot (IMPORTANT: run plotting cell right before this)
plt.savefig("outputs/figures/model_comparison.png")

print("All outputs saved!")

In [ ]:
from google.colab import files

files.download("outputs/tables/model_comparison.csv")
files.download("outputs/tables/sample_predictions.csv")
files.download("outputs/figures/model_comparison.png")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("outputs/figures", exist_ok=True)

# Rebuild comparison dataframe
comparison_df = pd.DataFrame([
    {
        "model": "Pretrained BERT QA",
        "exact_match": 84.0,
        "f1": 90.636232
    },
    {
        "model": "Fine-Tuned DistilBERT QA",
        "exact_match": 71.0,
        "f1": 80.316667
    }
])

comparison_long = comparison_df.melt(
    id_vars="model",
    value_vars=["exact_match", "f1"],
    var_name="metric",
    value_name="score"
)

plt.figure(figsize=(8, 5))
for metric in ["exact_match", "f1"]:
    subset = comparison_long[comparison_long["metric"] == metric]
    plt.plot(subset["model"], subset["score"], marker="o", label=metric)

plt.ylabel("Score")
plt.title("Baseline vs Fine-Tuned QA Performance")
plt.legend()
plt.xticks(rotation=15)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/figures/model_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: outputs/figures/model_comparison.png")

In [ ]:
from google.colab import files
files.download("outputs/figures/model_comparison.png")